<font size="6" color='grey'> <b>
Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br>

---

<font size="5" color='grey'> <b>
M09 Aufgabe A1: Iterative Prompt-Optimierung mit DALL-E
</b></font> </br>

# Aufgabe

**Ziel:** Implementiere einen 5-fachen Iterationszyklus, der Prompts automatisch optimiert.

**Prozess pro Iteration:**
1. **Image Generation**: Erstelle ein Bild mit DALL-E basierend auf dem aktuellen Prompt
2. **Image Classification**: Analysiere das generierte Bild mit GPT-4o-mini und optimiere den Prompt
3. **Feedback Loop**: Nutze die optimierte Beschreibung als neuer Prompt für die nächste Iteration

**Anforderungen:**
- 5 komplette Iterationszyklen
- Speichern aller generierten Bilder
- Dokumentation der Prompt-Entwicklung
- Vergleich: Original-Prompt vs. finale optimierte Version

# 1 | Setup & Installation

In [ ]:
#@title Umgebung einrichten { display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul
from genai_lib.utilities import check_environment, get_ipinfo, setup_api_keys, mprint, install_packages
setup_api_keys(['OPENAI_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# 2 | Importe

In [ ]:
import requests
import base64
from openai import OpenAI
from io import BytesIO
from PIL import Image as PILImage
from IPython.display import display, Image as IPImage, Markdown
import pandas as pd
from datetime import datetime
import os

# Initialize OpenAI client
client = OpenAI()

# 3 | Vorbereitung & Verzeichnis-Setup

In [ ]:
# Erstelle Output-Verzeichnis für Bilder
output_dir = "dalle_iterations"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

mprint("## Verzeichnis erstellt")
mprint(f"**Output-Dir:** {output_dir}")
print()

# 4 | Initial-Prompt definieren

In [ ]:
# Startprompt - bewusst einfach und vage
initial_prompt = "A magical forest with trees and light"

mprint("## Initial-Prompt")
mprint(f"`{initial_prompt}`")
print()

# 5 | Hilfsfunktionen

In [ ]:
def generate_image(prompt, iteration_num):
    """Generiert ein Bild mit DALL-E 3 basierend auf einem Prompt."""
    try:
        response = client.images.generate(
            model="dall-e-3",
            prompt=prompt,
            size="1024x1024",
            quality="standard",
            n=1
        )
        
        img_url = response.data[0].url
        img_data = requests.get(img_url)
        img = PILImage.open(BytesIO(img_data.content))
        
        filename = f"{output_dir}/iteration_{iteration_num}_generated.png"
        img.save(filename)
        
        return img, filename
    except Exception as e:
        print(f"Fehler bei Bildgenerierung: {str(e)}")
        return None, None

In [ ]:
def classify_and_optimize_prompt(image_path, current_prompt, iteration_num):
    """Klassifiziert ein Bild mit GPT-4o-mini und erstellt einen optimierten Prompt."""
    try:
        with open(image_path, "rb") as image_file:
            encoded_image = base64.b64encode(image_file.read()).decode("utf-8")
        
        image_url = f"data:image/png;base64,{encoded_image}"
        
        analysis_prompt = f"""Analysiere dieses Bild detailliert und erstelle einen verbesserten Prompt.

Aktuelle Prompt-Version: {current_prompt}

Deine Aufgabe:
1. Beschreibe was du im Bild siehst (Farben, Komposition, Details, Stil, Stimmung)
2. Identifiziere fehlende oder unterentwickelte Elemente
3. Erstelle einen DETAILLIERTEREN und SPEZIFISCHEREN Prompt fuer die naechste Iteration

Format der Antwort:
**Analyse:** [Deine Beobachtungen]
**Verbesserungen:** [3 Punkte]
**Optimierter Prompt:** [Neuer Prompt]"""
        
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": analysis_prompt},
                        {"type": "image_url", "image_url": {"url": image_url}}
                    ]
                }
            ],
            max_tokens=1000
        )
        
        full_response = response.choices[0].message.content
        
        if "Optimierter Prompt" in full_response:
            parts = full_response.split("Optimierter Prompt")
            if len(parts) > 1:
                optimized = parts[1].strip()
                if ":" in optimized:
                    optimized = optimized.split(":", 1)[1].strip()
                optimized_prompt = optimized.strip()
            else:
                optimized_prompt = full_response
        else:
            optimized_prompt = full_response
        
        return full_response, optimized_prompt
    except Exception as e:
        print(f"Fehler bei Bildklassifizierung: {str(e)}")
        return None, None

In [ ]:
def display_iteration_results(iteration_num, prompt, image, analysis):
    """Zeigt die Ergebnisse einer Iteration an."""
    mprint(f"## Iteration {iteration_num}")
    mprint(f"**Prompt:** `{prompt}`")
    
    if image:
        img_resized = image.resize((512, 512))
        display(img_resized)
    
    mprint(f"**Analyse von GPT-4o-mini:**")
    if analysis:
        display(Markdown(analysis))
    
    mprint("---\n")

# 6 | 5-facher Iterationszyklus

In [ ]:
iteration_history = []
current_prompt = initial_prompt

mprint("## Iterativer Prompt-Optimierungszyklus")
mprint(f"Start-Prompt: `{initial_prompt}`\n")
print()

for iteration in range(1, 6):
    print(f"\n{'='*60}")
    print(f"ITERATION {iteration}/5")
    print(f"{'='*60}\n")
    
    print(f"[Schritt 1] Generiere Bild mit DALL-E...")
    image, image_path = generate_image(current_prompt, iteration)
    
    if image is None:
        print(f"Fehler bei Iteration {iteration}")
        break
    
    print(f"OK: Bild gespeichert: {image_path}")
    
    print(f"\n[Schritt 2] Analysiere Bild mit GPT-4o-mini...")
    analysis, optimized_prompt = classify_and_optimize_prompt(image_path, current_prompt, iteration)
    
    if optimized_prompt is None:
        print(f"Fehler bei Bildanalyse fuer Iteration {iteration}")
        break
    
    print(f"OK: Prompt optimiert")
    
    iteration_history.append({
        "iteration": iteration,
        "prompt_used": current_prompt,
        "image_path": image_path,
        "analysis": analysis,
        "next_prompt": optimized_prompt
    })
    
    display_iteration_results(iteration, current_prompt, image, analysis)
    
    if iteration < 5:
        current_prompt = optimized_prompt[:2000]
        print(f"[Schritt 3] Naechster Prompt vorbereitet fuer Iteration {iteration + 1}...")

print(f"\n{'='*60}")
print(f"ALLE 5 ITERATIONEN ABGESCHLOSSEN")
print(f"{'='*60}\n")

# 7 | Zusammenfassung & Vergleich

In [ ]:
mprint("## Prompt-Entwicklung ueber 5 Iterationen")
mprint("")

for i, history_item in enumerate(iteration_history, 1):
    mprint(f"### Iteration {i}")
    mprint(f"**Prompt:** `{history_item['prompt_used']}`")
    mprint(f"**Bild:** {history_item['image_path']}")
    mprint(f"**Nächster Prompt:** `{history_item['next_prompt'][:150]}...`")
    mprint("")

# 8 | Original vs. Final

In [ ]:
if iteration_history:
    original_prompt = iteration_history[0]["prompt_used"]
    final_prompt = iteration_history[-1]["next_prompt"]
    
    mprint("## Original vs. Finale Prompt-Version")
    mprint("")
    mprint(f"**Original:** `{original_prompt}`")
    mprint(f"Länge: {len(original_prompt)} Zeichen")
    mprint("")
    mprint(f"**Final:** `{final_prompt}`")
    mprint(f"Länge: {len(final_prompt)} Zeichen")
    mprint(f"**Verbesserung:** +{len(final_prompt) - len(original_prompt)} Zeichen")

# 9 | Ergebnisse exportieren

In [ ]:
export_data = []

for i, history_item in enumerate(iteration_history, 1):
    export_data.append({
        "Iteration": i,
        "Prompt_verwendet": history_item["prompt_used"],
        "Prompt_laenge": len(history_item["prompt_used"]),
        "Bild_pfad": history_item["image_path"],
        "Naechster_Prompt": history_item["next_prompt"],
        "Naechster_Prompt_laenge": len(history_item["next_prompt"])
    })

df = pd.DataFrame(export_data)
csv_filename = f"{output_dir}/iteration_summary.csv"
df.to_csv(csv_filename, index=False)

mprint("## Ergebnisse exportiert")
mprint(f"**CSV-Datei:** {csv_filename}")
mprint(f"**Bilder-Verzeichnis:** {output_dir}")
mprint(f"**Gesamt-Bilder:** {len(iteration_history)}")